# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

This notebook compares two retrieval pipelines over **the same PDF, chunks, questions, and answer-generation model**.

- **Pipeline 1 — Baseline:** dense retrieval top 10 → keep top 3 → generate answer.
- **Pipeline 2 — Re-ranking:** dense retrieval top 10 → cross-encoder scores all 10 → keep re-ranked top 3 → generate answer.
- It saves intermediate retrieval evidence, ranks, scores, answers, timing logs, evaluation tables, plots, and a run summary to Google Drive.
- It resumes completed question/pipeline pairs instead of redoing them.

**Important evaluation note:** answer metrics need reference answers; retrieval metrics such as Recall@10 and nDCG@10 need chunk relevance judgments. The notebook will not invent these labels. Fill in `QUESTIONS` after inspecting your chosen PDF and its chunks.

Use this as an experiment and learning aid. Inspect the outputs and write the final report in your own words.

In [ ]:
# CELL 1 — Install the required libraries
# Run once at the start of the Colab session. If Colab asks to restart the runtime,
# restart it and continue from CELL 2.

!pip -q install pymupdf sentence-transformers transformers accelerate bitsandbytes scikit-learn pandas matplotlib seaborn rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s eta 0:00:00


In [ ]:
# CELL 2 — Mount Google Drive and configure the experiment

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, csv, time, logging, platform, subprocess, re, gc
from datetime import datetime, timezone

# CHANGE THIS to the exact path of your selected PDF in Google Drive.
PDF_PATH = "/content/drive/MyDrive/Namrata-IMP-Docs/IIITH/LMA_Project_Phase1_data/LMA_Assignment1/nasa_systems_engineering_handbook_0.pdf"
# All outputs are kept together so that results survive a Colab runtime reset.
OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model choices. These are deliberately moderate in size for both T4 and A100.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATION_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Retrieval/chunking hyperparameters.
CHUNK_SIZE_CHARS = 1400
CHUNK_OVERLAP_CHARS = 220
INITIAL_TOP_K = 10       # Candidate count from dense retrieval
FINAL_TOP_K = 3          # Number of chunks sent to the LLM
EMBEDDING_BATCH_SIZE = 32
RERANK_BATCH_SIZE = 8

# Generation settings: fixed for both approaches to make the comparison fair.
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0
USE_4BIT_QUANTIZATION = True

# Set to True only if you intentionally want to ignore saved outputs and rerun.
FORCE_RERUN = False

# Logging to both the notebook output and a persistent file.
LOG_PATH = OUTPUT_DIR / "run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler()]
)
logger = logging.getLogger("rag_assignment")

print("Output directory:", OUTPUT_DIR)
print("PDF exists:", Path(PDF_PATH).exists())
if not Path(PDF_PATH).exists():
    print("ACTION REQUIRED: upload your PDF to Google Drive and update PDF_PATH above.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/RAG_Assignment/outputs
PDF exists: True


## CELL 3 — Define the evaluation questions

Write questions that can be answered from the selected PDF. Include at least two **realistic failure cases** where a keyword-overlap distractor, exception, negation, or a chunk initially ranked 4–10 may matter.

For defensible retrieval metrics, `relevant_chunk_ids` must list the chunk IDs that genuinely contain evidence for the question. You can leave it empty initially; in that case retrieval metrics are reported as unavailable rather than fabricated.

After running the chunking/inspection cell below, inspect the chunks and fill in the relevant IDs. `reference_answer` should be a concise answer grounded in the PDF.

Use unique `id` values. Six to ten carefully checked questions is a practical starting point.

In [ ]:

# CELL 3A — Questions grounded in the NASA Systems Engineering Handbook
#
# IMPORTANT:
# 1. Verify the reference answers against the PDF before running.
# 2. Find the actual chunk IDs containing each answer and replace [].
# 3. Page numbers in comments refer to the handbook's printed page numbers.
# 4. A failure_type is a test hypothesis; confirm actual retrieval failures
#    from your baseline and reranking results.

QUESTIONS = [
    {
        "id": "Q01",
        "question": (
            "What are the four System Design Processes described by NASA, "
            "and what is their overall purpose?"
        ),
        "reference_answer": (
            "The four System Design Processes are Stakeholder Expectations "
            "Definition, Technical Requirements Definition, Logical "
            "Decomposition, and Design Solution Definition. Together, they "
            "define stakeholder expectations, establish technical requirements, "
            "decompose requirements into logical and behavioral models, and "
            "develop a design solution that satisfies the expectations."
        ),
        # Answer location: Section 2.0, Fundamentals of Systems Engineering,
        # around printed page 6; Figure 2.1-1 and the paragraph on System
        # Design Processes immediately below it.
        "relevant_chunk_ids": ["p0016_c000"],
        "failure_type": "ordinary"
    },
    {
        "id": "Q02",
        "question": (
            "What is the difference between the Product Verification Process "
            "and the Product Validation Process?"
        ),
        "reference_answer": (
            "Verification demonstrates that an end product conforms to its "
            "design solution requirements. Validation confirms that the "
            "verified end product fulfills its intended use in its intended "
            "environment and satisfies baselined stakeholder expectations."
        ),
        # Answer location: Appendix B, Glossary, entries for
        # 'Product Verification Process' and 'Product Validation Process',
        # around printed page 189.
        "relevant_chunk_ids": ["p0021_c00", "p0021_c001"],
        "failure_type": "keyword_trap"
    },
    {
        "id": "Q03",
        "question": (
            "What characteristics and requirements are described by NASA's "
            "product baseline?"
        ),
        "reference_answer": (
            "The product baseline describes the detailed physical or form, "
            "fit, and function characteristics of a configuration item; "
            "the selected functional characteristics designated for production "
            "acceptance testing; and the production acceptance test requirements."
        ),
        # Answer location: Section 6.5.1.2.3, Manage Configuration Change
        # Control, in the discussion of types/evolution of baselines,
        # around printed page 148.
        "relevant_chunk_ids": ["p0157_c001"],
        "failure_type": "ordinary"
    },
    {
        "id": "Q04",
        "question": (
            "What are the main steps in NASA's normal configuration change "
            "control process, from submitting a change request to closing it?"
        ),
        "reference_answer": (
            "The process includes preparing and submitting a change request; "
            "logging and screening it; releasing it for review; collecting "
            "review comments; preparing a decision package; obtaining the "
            "Configuration Control Board's disposition and assigning actions "
            "as needed; completing assigned actions; implementing approved "
            "changes and finalizing documentation; verifying the release "
            "package; and releasing the final documentation and closing "
            "the request."
        ),
        # Answer location: Section 6.5.1.2.3, Figure 6.5-4,
        # 'Typical Change Control Process', around printed page 149.
        "relevant_chunk_ids": ["p0158_c001"],
        "failure_type": "rank_shift"
    },
    {
        "id": "Q05",
        "question": (
            "According to NASA, how does a major engineering change differ "
            "from a minor engineering change?"
        ),
        "reference_answer": (
            "A major engineering change significantly affects the baseline "
            "configuration documentation and may affect delivered products, "
            "baseline specifications, cost, safety, compatibility with "
            "interfacing products, or operator and maintenance training. "
            "A minor change modifies or corrects documentation or processes "
            "without affecting product interchangeability."
        ),
        # Answer location: Section 6.5.1.2.3, under 'Types of Configuration
        # Management Changes', around printed page 149.
        "relevant_chunk_ids": ["p0159_c000"],
        "failure_type": "exception"
    },
    {
        "id": "Q06",
        "question": (
            "Who chairs NASA's internal Configuration Control Board, and "
            "what is the board responsible for in configuration change control?"
        ),
        "reference_answer": (
            "The internal NASA Configuration Control Board is chaired by "
            "someone with program or project change authority. The board "
            "members represent stakeholders authorized to commit their teams. "
            "The board participates in the disposition of change requests "
            "and assignment of actions as part of the change control process."
        ),
        # Answer location: Section 6.5.1.2.3, Manage Configuration Change
        # Control, and Figure 6.5-4, around printed pages 148–149.
        "relevant_chunk_ids": ["p0158_c000"],
        "failure_type": "keyword_trap"
    },
    {
        "id": "Q07",
        "question": (
            "At what review is the allocated baseline typically established, "
            "and what does this baseline specify?"
        ),
        "reference_answer": (
            "The allocated baseline is typically established at the "
            "Preliminary Design Review (PDR). It describes the functional "
            "and interface characteristics allocated from higher-level "
            "requirements and the verification required to demonstrate "
            "that those characteristics have been achieved."
        ),
        # Answer location: Appendix B, Glossary, entry for 'Allocated
        # Baseline (Phase C)', around printed page 177.
        "relevant_chunk_ids": ["p0157_c000"],
        "failure_type": "rank_shift"
    },
    {
        "id": "Q08",
        "question": (
            "When is the as-deployed baseline established, and what does "
            "it indicate about the system?"
        ),
        "reference_answer": (
            "The as-deployed baseline occurs at the Operational Readiness "
            "Review (ORR). At that point, the design is considered functional "
            "and ready for flight, and the documentation incorporates the "
            "changes made up to that stage."
        ),
        # Answer location: Section 6.5.1.2.3, discussion of baseline
        # evolution, around printed page 148.
        "relevant_chunk_ids": ["p0158_c000"],
        "failure_type": "ordinary"
    },
    {
        "id": "Q09",
        "question": (
            "Why does NASA use configuration change management, and what "
            "activities are involved in managing an approved design change?"
        ),
        "reference_answer": (
            "Configuration change management controls approved designs and "
            "the implementation of approved changes. It involves systematically "
            "proposing, justifying, and evaluating changes, incorporating "
            "approved changes, and verifying that implementation."
        ),
        # Answer location: Section 6.5.1.2.3, opening paragraph under
        # 'Manage Configuration Change Control', around printed page 148.
        "relevant_chunk_ids": ["p0158_c000"],
        "failure_type": "negation"
    },
    {
        "id": "Q10",
        "question": (
            "What factors should be considered when deciding the level of "
            "simulation fidelity required for product verification during "
            "operations?"
        ),
        "reference_answer": (
            "Factors include runtime constraints, the level of detail "
            "required, the level of system integration, and environmental "
            "effects. The necessary fidelity depends on the nature and "
            "criticality of the product and the verification activity."
        ),
        # Answer location: Appendix T, Section T.3.2.2, Product Verification,
        # around printed page 257.
        "relevant_chunk_ids": ["p0267_c000", "p0267_c001"],
        "failure_type": "rank_shift"
    }
]


In [ ]:
# CELL 4 — Core functions: PDF extraction, chunking, models, retrieval, generation and metrics

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def append_jsonl(path, record):
    """Append one JSON record immediately, so progress survives interruptions."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

def read_jsonl(path):
    """Read a JSONL file, ignoring a final incomplete line if an interruption occurred."""
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                if line.strip():
                    rows.append(json.loads(line))
            except json.JSONDecodeError:
                logger.warning("Ignoring malformed/incomplete JSONL line in %s", path)
    return rows

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

def get_hardware_info():
    info = {"python": platform.python_version(), "platform": platform.platform()}
    try:
        import torch
        info["torch_version"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            info["gpu_name"] = torch.cuda.get_device_name(0)
            props = torch.cuda.get_device_properties(0)
            info["gpu_total_memory_gb"] = round(props.total_memory / (1024**3), 2)
    except Exception as e:
        info["torch_info_error"] = str(e)
    return info

def gpu_memory_snapshot():
    """Return current and peak allocated/reserved CUDA memory when CUDA is available."""
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            return {
                "allocated_gb": round(torch.cuda.memory_allocated() / (1024**3), 3),
                "reserved_gb": round(torch.cuda.memory_reserved() / (1024**3), 3),
                "peak_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024**3), 3),
                "peak_reserved_gb": round(torch.cuda.max_memory_reserved() / (1024**3), 3),
            }
    except Exception:
        pass
    return {}

def extract_pdf_pages(pdf_path):
    """Extract selectable text from each page and preserve page numbers.

    Scanned/image-only PDFs may need OCR; this notebook intentionally warns if little
    text is found rather than silently pretending extraction worked.
    """
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append({"page": page_num, "text": text})
    doc.close()
    total_chars = sum(len(p["text"]) for p in pages)
    if total_chars < 1000:
        raise ValueError(
            f"Only {total_chars} text characters were extracted. The PDF may be scanned "
            "or image-only; use an OCR-capable version or add OCR before continuing."
        )
    return pages

def chunk_pages(pages, chunk_size=CHUNK_SIZE_CHARS, overlap=CHUNK_OVERLAP_CHARS):
    """Split page text into overlapping character windows and keep page provenance.

    Character windows are simple and transparent. They may split a sentence; inspect
    several chunks and adjust size/overlap if necessary.
    """
    if overlap >= chunk_size:
        raise ValueError("CHUNK_OVERLAP_CHARS must be smaller than CHUNK_SIZE_CHARS.")
    chunks = []
    for page_item in pages:
        text = re.sub(r"\s+", " ", page_item["text"]).strip()
        start, local_idx = 0, 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            piece = text[start:end].strip()
            if piece:
                chunks.append({
                    "chunk_id": f"p{page_item['page']:04d}_c{local_idx:03d}",
                    "page": page_item["page"],
                    "start_char": start,
                    "end_char": end,
                    "text": piece
                })
                local_idx += 1
            if end == len(text):
                break
            start = end - overlap
    return chunks

def save_chunks(chunks):
    path = OUTPUT_DIR / "document_chunks.jsonl"
    path.write_text(
        "".join(json.dumps(c, ensure_ascii=False) + "\n" for c in chunks),
        encoding="utf-8"
    )
    return path

def load_or_build_chunks():
    """Reuse chunks only if they were built from the currently configured PDF path."""
    if not Path(PDF_PATH).exists():
        raise FileNotFoundError(f"PDF not found: {PDF_PATH}")
    pages = extract_pdf_pages(PDF_PATH)
    chunks = chunk_pages(pages)
    save_chunks(chunks)
    write_json(OUTPUT_DIR / "document_metadata.json", {
        "pdf_path": PDF_PATH,
        "page_count_with_text": len(pages),
        "chunk_count": len(chunks),
        "total_extracted_characters": sum(len(p["text"]) for p in pages),
        "chunk_size_chars": CHUNK_SIZE_CHARS,
        "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
        "created_at_utc": utc_now()
    })
    return chunks

def print_chunk_samples(chunks, n=5, max_chars=700):
    """Inspect these samples before you create relevance judgments."""
    for chunk in chunks[:n]:
        print(f"\n[{chunk['chunk_id']} | page {chunk['page']}]")
        print(chunk["text"][:max_chars])

def load_models():
    """Load embedding, cross-encoder, and local generation models.

    The generation model is quantized to 4-bit when requested and supported.
    On a T4, 4-bit loading is recommended for this 3B model.
    """
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info("Loading embedding model: %s", EMBEDDING_MODEL_NAME)
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

    logger.info("Loading cross-encoder: %s", CROSS_ENCODER_NAME)
    reranker = CrossEncoder(CROSS_ENCODER_NAME, device=device)

    logger.info("Loading generation model: %s", GENERATION_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    model_kwargs = {"device_map": "auto"}
    if device == "cuda" and USE_4BIT_QUANTIZATION:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        model_kwargs["quantization_config"] = quant_config
        model_kwargs["torch_dtype"] = torch.float16
    else:
        model_kwargs["torch_dtype"] = torch.float32 if device == "cpu" else torch.float16

    generator = AutoModelForCausalLM.from_pretrained(
        GENERATION_MODEL_NAME, **model_kwargs
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return embedder, reranker, tokenizer, generator

def build_embedding_matrix(chunks, embedder):
    """Embed each chunk once; both approaches share this exact matrix."""
    texts = [c["text"] for c in chunks]
    vectors = embedder.encode(
        texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save(OUTPUT_DIR / "chunk_embeddings.npy", vectors)
    return vectors

def dense_retrieve(question, chunks, embedding_matrix, embedder, top_k=INITIAL_TOP_K):
    """Return top-k chunks sorted by cosine similarity.

    Embeddings are normalized, so their dot product is cosine similarity.
    """
    q_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = np.dot(embedding_matrix, q_vec)
    top_indices = np.argsort(scores)[::-1][:min(top_k, len(chunks))]
    return [
        {**chunks[int(i)], "dense_rank": rank, "dense_score": float(scores[int(i)])}
        for rank, i in enumerate(top_indices, start=1)
    ]

def rerank_candidates(question, candidates, reranker):
    """Score each (question, chunk) pair jointly with the cross-encoder."""
    pairs = [(question, c["text"]) for c in candidates]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    scored = []
    for c, score in zip(candidates, scores):
        scored.append({**c, "cross_encoder_score": float(score)})
    scored.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
    for rank, c in enumerate(scored, start=1):
        c["reranked_rank"] = rank
    return scored

def make_prompt(question, selected_chunks):
    """Keep this prompt identical across both approaches; only the chosen chunks differ."""
    context = "\n\n".join(
        f"[Chunk {c['chunk_id']} | page {c['page']}]\n{c['text']}"
        for c in selected_chunks
    )
    return f'''You answer questions using only the supplied document excerpts.
If the excerpts do not contain enough information, say "The provided excerpts do not specify this."
Do not add facts from outside the excerpts. Be concise and mention relevant rule/section details when present.

DOCUMENT EXCERPTS:
{context}

QUESTION:
{question}

ANSWER:'''

def generate_answer(question, selected_chunks, tokenizer, generator):
    """Generate a grounded answer from the selected top-3 chunks."""
    import torch
    prompt = make_prompt(question, selected_chunks)
    messages = [
        {"role": "system", "content": "You are a careful document question-answering assistant. Use only supplied excerpts."},
        {"role": "user", "content": prompt}
    ]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=6000)
    # Put inputs on the embedding device; device_map='auto' places model layers as needed.
    input_device = generator.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": (TEMPERATURE > 0),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    # Only pass temperature when sampling is enabled. With temperature=0, greedy
    # decoding is used and the temperature argument is unnecessary.
    if TEMPERATURE > 0:
        generation_kwargs["temperature"] = TEMPERATURE

    with torch.inference_mode():
        output = generator.generate(**inputs, **generation_kwargs)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, prompt

def normalize_text(s):
    s = (s or "").lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def token_f1(prediction, reference):
    """Token-overlap F1; useful as a simple baseline, not a complete semantic metric."""
    p = normalize_text(prediction).split()
    r = normalize_text(reference).split()
    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0
    from collections import Counter
    overlap = sum((Counter(p) & Counter(r)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(r)
    return 2 * precision * recall / (precision + recall)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, cid in enumerate(retrieved_ids, start=1):
        if cid in set(relevant_ids):
            return 1.0 / rank
    return 0.0

def retrieval_metrics(retrieved_ids, relevant_ids, k):
    """Compute retrieval metrics only when relevant chunk IDs were provided and verified."""
    if not relevant_ids:
        return {"recall_at_k": None, "mrr_at_k": None, "ndcg_at_k": None}
    relevant = set(relevant_ids)
    ranked = retrieved_ids[:k]
    hits = [1 if cid in relevant else 0 for cid in ranked]
    recall = len(set(ranked) & relevant) / len(relevant)
    mrr = reciprocal_rank(ranked, relevant)
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(hits))
    ideal_hits = [1] * min(len(relevant), k)
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_hits))
    ndcg = dcg / idcg if idcg else 0.0
    return {"recall_at_k": recall, "mrr_at_k": mrr, "ndcg_at_k": ndcg}

def validate_questions(questions):
    if not questions:
        raise ValueError(
            "QUESTIONS is empty. Add questions grounded in the PDF, concise reference_answer "
            "values, and verified relevant_chunk_ids where possible before running."
        )
    ids = [q.get("id") for q in questions]
    if any(not x for x in ids) or len(ids) != len(set(ids)):
        raise ValueError("Every question must have a unique, non-empty id.")
    for q in questions:
        for key in ("question", "reference_answer"):
            if not q.get(key):
                raise ValueError(f"{q.get('id')}: missing {key}.")
        q.setdefault("relevant_chunk_ids", [])
        q.setdefault("failure_type", "ordinary")
    return questions

In [ ]:
# CELL 5 — One main function runs BOTH pipelines, evaluates them, logs timing, and plots results.

def run_full_experiment(questions=QUESTIONS, force_rerun=FORCE_RERUN):
    """Run the baseline and re-ranking experiments end-to-end.

    Resume behavior:
    - Each question/pipeline result is appended to its own JSONL file immediately.
    - If a completed record already exists, it is reused unless force_rerun=True.
    - Evaluation and plots are rebuilt from saved records at the end.
    """
    import torch

    questions = validate_questions(questions)
    if not Path(PDF_PATH).exists():
        raise FileNotFoundError(f"PDF_PATH does not exist: {PDF_PATH}")

    run_started = time.perf_counter()
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    hardware = get_hardware_info()
    write_json(OUTPUT_DIR / "hardware_info.json", hardware)
    write_json(OUTPUT_DIR / "config.json", {
        "run_id": run_id,
        "pdf_path": PDF_PATH,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "cross_encoder_model": CROSS_ENCODER_NAME,
        "generation_model": GENERATION_MODEL_NAME,
        "chunk_size_chars": CHUNK_SIZE_CHARS,
        "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
        "initial_top_k": INITIAL_TOP_K,
        "final_top_k": FINAL_TOP_K,
        "embedding_batch_size": EMBEDDING_BATCH_SIZE,
        "rerank_batch_size": RERANK_BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "quantization_4bit": USE_4BIT_QUANTIZATION,
        "hardware": hardware,
        "created_at_utc": utc_now()
    })

    # Load or recreate document chunks. Chunk IDs preserve page provenance.
    t0 = time.perf_counter()
    chunks = load_or_build_chunks()
    chunk_seconds = time.perf_counter() - t0
    logger.info("Loaded PDF and created %d chunks in %.2f sec", len(chunks), chunk_seconds)

    # Save all question definitions so the experiment can be reproduced.
    write_json(OUTPUT_DIR / "evaluation_questions.json", questions)

    # Load models and compute shared embeddings.
    t0 = time.perf_counter()
    embedder, reranker, tokenizer, generator = load_models()
    model_load_seconds = time.perf_counter() - t0
    logger.info("Model loading took %.2f sec", model_load_seconds)

    t0 = time.perf_counter()
    emb_path = OUTPUT_DIR / "chunk_embeddings.npy"
    if emb_path.exists() and not force_rerun:
        embedding_matrix = np.load(emb_path)
        if len(embedding_matrix) != len(chunks):
            logger.warning("Embedding count differs from chunks; rebuilding embeddings.")
            embedding_matrix = build_embedding_matrix(chunks, embedder)
    else:
        embedding_matrix = build_embedding_matrix(chunks, embedder)
    embedding_seconds = time.perf_counter() - t0
    logger.info("Embedding/index preparation took %.2f sec", embedding_seconds)

    baseline_path = OUTPUT_DIR / "baseline_results.jsonl"
    reranked_path = OUTPUT_DIR / "reranked_results.jsonl"
    retrieval_path = OUTPUT_DIR / "retrieval_logs.jsonl"
    rerank_log_path = OUTPUT_DIR / "reranking_logs.jsonl"
    timing_path = OUTPUT_DIR / "timings.jsonl"

    baseline_existing = {} if force_rerun else {
        r["question_id"]: r for r in read_jsonl(baseline_path) if r.get("status") == "ok"
    }
    reranked_existing = {} if force_rerun else {
        r["question_id"]: r for r in read_jsonl(reranked_path) if r.get("status") == "ok"
    }

    # force_rerun starts new log files, avoiding duplicate old and new results.
    if force_rerun:
        for p in [baseline_path, reranked_path, retrieval_path, rerank_log_path, timing_path]:
            if p.exists():
                p.unlink()

    # Reset peak CUDA counters so memory measurements relate to this run.
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for index, q in enumerate(questions, start=1):
        qid, query = q["id"], q["question"]
        logger.info("Question %d/%d: %s", index, len(questions), qid)

        # The initial top-10 retrieval is shared by both pipelines.
        t0 = time.perf_counter()
        candidates = dense_retrieve(query, chunks, embedding_matrix, embedder, INITIAL_TOP_K)
        dense_seconds = time.perf_counter() - t0
        append_jsonl(retrieval_path, {
            "timestamp_utc": utc_now(), "question_id": qid, "question": query,
            "candidates": [
                {k: c[k] for k in ["chunk_id", "page", "dense_rank", "dense_score", "text"]}
                for c in candidates
            ]
        })

        # Pipeline 1: use the dense top-3 without re-ranking.
        if qid not in baseline_existing:
            t0 = time.perf_counter()
            selected_baseline = candidates[:FINAL_TOP_K]
            try:
                answer1, prompt1 = generate_answer(query, selected_baseline, tokenizer, generator)
                baseline_record = {
                    "timestamp_utc": utc_now(), "question_id": qid, "question": query,
                    "reference_answer": q["reference_answer"],
                    "failure_type": q.get("failure_type", "ordinary"),
                    "status": "ok", "answer": answer1, "prompt": prompt1,
                    "selected_chunks": [
                        {k: c[k] for k in ["chunk_id", "page", "dense_rank", "dense_score", "text"]}
                        for c in selected_baseline
                    ],
                    "dense_retrieval_seconds": dense_seconds,
                    "generation_seconds": time.perf_counter() - t0,
                    "answer_token_f1": token_f1(answer1, q["reference_answer"]),
                    "retrieval_metrics_top10": retrieval_metrics(
                        [c["chunk_id"] for c in candidates], q["relevant_chunk_ids"], INITIAL_TOP_K
                    ),
                    "retrieval_metrics_top3": retrieval_metrics(
                        [c["chunk_id"] for c in selected_baseline], q["relevant_chunk_ids"], FINAL_TOP_K
                    )
                }
                append_jsonl(baseline_path, baseline_record)
                baseline_existing[qid] = baseline_record
            except Exception as e:
                logger.exception("Baseline failed for %s", qid)
                append_jsonl(baseline_path, {
                    "timestamp_utc": utc_now(), "question_id": qid, "question": query,
                    "status": "error", "error": repr(e)
                })

        # Pipeline 2: re-score the same top-10 candidates, then use the new top-3.
        if qid not in reranked_existing:
            t0 = time.perf_counter()
            try:
                reranked_candidates = rerank_candidates(query, candidates, reranker)
                rerank_seconds = time.perf_counter() - t0
                selected_reranked = reranked_candidates[:FINAL_TOP_K]
                append_jsonl(rerank_log_path, {
                    "timestamp_utc": utc_now(), "question_id": qid, "question": query,
                    "candidates": [
                        {k: c[k] for k in [
                            "chunk_id", "page", "dense_rank", "dense_score",
                            "cross_encoder_score", "reranked_rank", "text"
                        ]} for c in reranked_candidates
                    ]
                })

                gen_t0 = time.perf_counter()
                answer2, prompt2 = generate_answer(query, selected_reranked, tokenizer, generator)
                generation_seconds = time.perf_counter() - gen_t0
                reranked_record = {
                    "timestamp_utc": utc_now(), "question_id": qid, "question": query,
                    "reference_answer": q["reference_answer"],
                    "failure_type": q.get("failure_type", "ordinary"),
                    "status": "ok", "answer": answer2, "prompt": prompt2,
                    "selected_chunks": [
                        {k: c[k] for k in [
                            "chunk_id", "page", "dense_rank", "dense_score",
                            "cross_encoder_score", "reranked_rank", "text"
                        ]} for c in selected_reranked
                    ],
                    "dense_retrieval_seconds": dense_seconds,
                    "reranking_seconds": rerank_seconds,
                    "generation_seconds": generation_seconds,
                    "answer_token_f1": token_f1(answer2, q["reference_answer"]),
                    "retrieval_metrics_top10": retrieval_metrics(
                        [c["chunk_id"] for c in candidates], q["relevant_chunk_ids"], INITIAL_TOP_K
                    ),
                    "retrieval_metrics_reranked_top3": retrieval_metrics(
                        [c["chunk_id"] for c in reranked_candidates[:FINAL_TOP_K]],
                        q["relevant_chunk_ids"], FINAL_TOP_K
                    )
                }
                append_jsonl(reranked_path, reranked_record)
                reranked_existing[qid] = reranked_record
            except Exception as e:
                logger.exception("Re-ranking pipeline failed for %s", qid)
                append_jsonl(reranked_path, {
                    "timestamp_utc": utc_now(), "question_id": qid, "question": query,
                    "status": "error", "error": repr(e)
                })

        append_jsonl(timing_path, {
            "timestamp_utc": utc_now(), "question_id": qid,
            "dense_retrieval_seconds": dense_seconds,
            "cuda_memory": gpu_memory_snapshot()
        })

        # Periodic lightweight checkpoint summary.
        write_json(OUTPUT_DIR / "progress.json", {
            "run_id": run_id, "last_question": qid,
            "questions_total": len(questions),
            "baseline_completed": len(baseline_existing),
            "reranked_completed": len(reranked_existing),
            "updated_at_utc": utc_now()
        })

    # Build evaluation tables using successful records from disk.
    baseline_records = {
        r["question_id"]: r for r in read_jsonl(baseline_path) if r.get("status") == "ok"
    }
    reranked_records = {
        r["question_id"]: r for r in read_jsonl(reranked_path) if r.get("status") == "ok"
    }

    rows = []
    rank_rows = []
    for q in questions:
        qid = q["id"]
        b = baseline_records.get(qid)
        r = reranked_records.get(qid)
        if not b or not r:
            continue
        b_ids = [c["chunk_id"] for c in b["selected_chunks"]]
        r_ids = [c["chunk_id"] for c in r["selected_chunks"]]
        rows.append({
            "question_id": qid,
            "failure_type": q.get("failure_type", "ordinary"),
            "baseline_answer": b["answer"],
            "reranked_answer": r["answer"],
            "reference_answer": q["reference_answer"],
            "baseline_token_f1": b["answer_token_f1"],
            "reranked_token_f1": r["answer_token_f1"],
            "baseline_top3_recall": b["retrieval_metrics_top3"]["recall_at_k"],
            "reranked_top3_recall": r["retrieval_metrics_reranked_top3"]["recall_at_k"],
            "baseline_dense_retrieval_seconds": b["dense_retrieval_seconds"],
            "baseline_generation_seconds": b["generation_seconds"],
            "rerank_seconds": r.get("reranking_seconds"),
            "reranked_generation_seconds": r["generation_seconds"],
            "baseline_top3_chunk_ids": "|".join(b_ids),
            "reranked_top3_chunk_ids": "|".join(r_ids),
        })
        # Rank shift data: show all initial candidates and where they ended up.
        rerank_log = next(
            (x for x in reversed(read_jsonl(rerank_log_path)) if x.get("question_id") == qid),
            None
        )
        if rerank_log:
            for c in rerank_log["candidates"]:
                rank_rows.append({
                    "question_id": qid,
                    "chunk_id": c["chunk_id"],
                    "page": c["page"],
                    "dense_rank": c["dense_rank"],
                    "reranked_rank": c["reranked_rank"],
                    "dense_score": c["dense_score"],
                    "cross_encoder_score": c["cross_encoder_score"],
                    "relevant": c["chunk_id"] in set(q.get("relevant_chunk_ids", []))
                })

    results_df = pd.DataFrame(rows)
    rank_df = pd.DataFrame(rank_rows)
    results_df.to_csv(OUTPUT_DIR / "answer_comparison_and_metrics.csv", index=False)
    rank_df.to_csv(OUTPUT_DIR / "rank_shift_analysis.csv", index=False)

    # Plots: only plot metrics that exist; retrieval metrics may be unavailable if
    # relevance labels have not been filled in.
    plot_paths = []
    if not results_df.empty:
        plt.figure(figsize=(10, 5))
        x = np.arange(len(results_df))
        width = 0.36
        plt.bar(x - width/2, results_df["baseline_token_f1"], width, label="Baseline")
        plt.bar(x + width/2, results_df["reranked_token_f1"], width, label="Re-ranked")
        plt.xticks(x, results_df["question_id"], rotation=45, ha="right")
        plt.ylabel("Token-overlap F1 (0–1)")
        plt.title("Answer overlap with reference answer")
        plt.legend()
        plt.tight_layout()
        p = OUTPUT_DIR / "answer_token_f1.png"
        plt.savefig(p, dpi=180); plt.close(); plot_paths.append(str(p))

        latency_cols = ["baseline_generation_seconds", "rerank_seconds", "reranked_generation_seconds"]
        latency_means = {
            "Baseline generation": results_df["baseline_generation_seconds"].mean(),
            "Cross-encoder re-ranking": results_df["rerank_seconds"].mean(),
            "Re-ranked generation": results_df["reranked_generation_seconds"].mean()
        }
        plt.figure(figsize=(8, 5))
        plt.bar(list(latency_means.keys()), list(latency_means.values()))
        plt.ylabel("Mean seconds per question")
        plt.title("Measured stage latency (mean)")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        p = OUTPUT_DIR / "latency_comparison.png"
        plt.savefig(p, dpi=180); plt.close(); plot_paths.append(str(p))

    if not rank_df.empty:
        plt.figure(figsize=(8, 6))
        for qid, group in rank_df.groupby("question_id"):
            plt.plot(group["dense_rank"], group["reranked_rank"], marker="o", linestyle="",
                     label=qid)
        plt.plot([1, INITIAL_TOP_K], [1, INITIAL_TOP_K], linestyle="--")
        plt.xlabel("Initial dense rank")
        plt.ylabel("Cross-encoder rank")
        plt.title("Candidate rank shifts (points below diagonal moved up)")
        plt.xlim(0.5, INITIAL_TOP_K + 0.5)
        plt.ylim(0.5, INITIAL_TOP_K + 0.5)
        plt.gca().invert_yaxis()
        plt.gca().invert_xaxis()
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        p = OUTPUT_DIR / "rank_shifts.png"
        plt.savefig(p, dpi=180); plt.close(); plot_paths.append(str(p))

    # Aggregate summary: don't label token-overlap F1 as semantic correctness.
    summary = {
        "run_id": run_id,
        "completed_at_utc": utc_now(),
        "hardware": hardware,
        "pdf_path": PDF_PATH,
        "chunk_count": len(chunks),
        "question_count": len(questions),
        "baseline_successful": len(baseline_records),
        "reranked_successful": len(reranked_records),
        "chunking_seconds": chunk_seconds,
        "model_loading_seconds": model_load_seconds,
        "embedding_preparation_seconds": embedding_seconds,
        "elapsed_seconds_this_invocation": time.perf_counter() - run_started,
        "cuda_memory_at_end": gpu_memory_snapshot(),
        "mean_baseline_token_f1": float(results_df["baseline_token_f1"].mean()) if not results_df.empty else None,
        "mean_reranked_token_f1": float(results_df["reranked_token_f1"].mean()) if not results_df.empty else None,
        "mean_baseline_generation_seconds": float(results_df["baseline_generation_seconds"].mean()) if not results_df.empty else None,
        "mean_reranking_seconds": float(results_df["rerank_seconds"].mean()) if not results_df.empty else None,
        "mean_reranked_generation_seconds": float(results_df["reranked_generation_seconds"].mean()) if not results_df.empty else None,
        "plots": plot_paths,
        "note": "Token-overlap F1 is lexical overlap, not a definitive semantic correctness score. Retrieval metrics require verified relevant_chunk_ids."
    }
    write_json(OUTPUT_DIR / "summary.json", summary)

    # Concise report scaffold with actual numbers. Use it as evidence, then write the
    # final assignment report yourself in your own words.
    report_lines = [
        "# RAG Baseline vs. Re-Ranking — Results Scaffold",
        "",
        f"- PDF: `{PDF_PATH}`",
        f"- Chunks: {len(chunks)}",
        f"- Questions: {len(questions)}",
        f"- GPU: {hardware.get('gpu_name', 'not detected')}",
        f"- Baseline successful questions: {len(baseline_records)}/{len(questions)}",
        f"- Re-ranked successful questions: {len(reranked_records)}/{len(questions)}",
        f"- Mean baseline token-overlap F1: {summary['mean_baseline_token_f1']}",
        f"- Mean re-ranked token-overlap F1: {summary['mean_reranked_token_f1']}",
        f"- Mean cross-encoder time/question: {summary['mean_reranking_seconds']}",
        "",
        "## Interpretation reminders",
        "- Token-overlap F1 measures lexical overlap with the reference, not full semantic correctness.",
        "- Retrieval metrics are available only for questions with verified relevant chunk IDs.",
        "- Use rank_shift_analysis.csv and retrieval_logs.jsonl to inspect rank changes.",
        "- Compare both generated answers against the document before claiming an improvement.",
        "- Explain observed failures and cases where re-ranking did not help."
    ]
    (OUTPUT_DIR / "report_results_scaffold.md").write_text("\n".join(report_lines), encoding="utf-8")

    logger.info("Experiment finished. Results saved to %s", OUTPUT_DIR)
    print("\nEXPERIMENT SUMMARY")
    print(json.dumps(summary, indent=2))
    print("\nFiles:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
    return {"summary": summary, "results": results_df, "rank_shifts": rank_df, "output_dir": str(OUTPUT_DIR)}

## CELL 6 — Inspect chunks and prepare questions (run before the experiment)

Run this cell after setting `PDF_PATH`. Inspect the chunks, then write `QUESTIONS` in CELL 3A. You can run this cell first without loading the language models.

For relevance labels:
1. Find which chunk(s) actually contain evidence for each question.
2. Put their exact `chunk_id` strings into `relevant_chunk_ids`.
3. If the evidence is spread across multiple chunks, include all chunks needed to answer the question.
4. If you have not verified relevance labels, use `[]`; the notebook will skip retrieval metrics for that question.

In [ ]:
# CELL 6A — Inspect the document and sample chunks (does not load the LLM)

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(f"Update PDF_PATH first: {PDF_PATH}")

chunks_for_review = load_or_build_chunks()
print(f"Extracted {len(chunks_for_review)} chunks.")
print_chunk_samples(chunks_for_review, n=min(8, len(chunks_for_review)))

Extracted 770 chunks.

[p0001_c000 | page 1]
National Aeronautics and Space Administration NASA SYSTEMS ENGINEERING HANDBOOK design test integrate fly www.nasa.gov

[p0002_c000 | page 2]
NASA SP-2016-6105 Rev2 supersedes SP-2007-6105 Rev 1 dated December, 2007. Cover photos: Top left: In this photo, engineers led by researcher Greg Gatlin have sprayed fluorescent oil on a 5.8 percent scale model of a futuristic hybrid wing body during tests in the 14- by 22-Foot Subsonic Wind Tunnel at NASA’s Langley Research Center in Hampton, VA. The oil helps researchers “see” the flow patterns when air passes over and around the model. (NASA Langley/ Preston Martin) Top right: Water impact test of a test version of the Orion spacecraft took place on August 24, 2016, at NASA Langley Research Center (NASA) Bottom left: two test mirror segments are placed onto the support structure that wil

[p0003_c000 | page 3]
iii NASA SYSTEMS ENGINEERING HANDBOOK Table of Contents Preface. . . . . . . . . . . . . 

## CELL 7 — Run both approaches and evaluate

Once `QUESTIONS` is populated and you have reviewed the PDF/chunks, run this cell. This is the **single function call** that runs both approaches, evaluates them, and produces plots.

Start with 3 questions as a pilot. If the pipeline and outputs look correct, expand to 6–10 well-grounded questions. A local 3B model may take several minutes per question depending on GPU, prompt length, and model loading time. Colab does not guarantee a runtime duration; checkpoints are saved after each question/pipeline.

In [ ]:
# CELL 7A — One call to run the full experiment

experiment = run_full_experiment(questions=QUESTIONS, force_rerun=FORCE_RERUN)

# Preview the principal comparison table.
display(experiment["results"])
display(experiment["rank_shifts"].head(20))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]


EXPERIMENT SUMMARY
{
  "run_id": "20261019_090829",
  "completed_at_utc": "2026-10-19T09:11:33.128720+00:00",
  "hardware": {
    "python": "3.13.15",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "torch_version": "2.11.0+cu130",
    "cuda_available": true,
    "gpu_name": "Tesla T4",
    "gpu_total_memory_gb": 14.56
  },
  "pdf_path": "/content/drive/MyDrive/Namrata-IMP-Docs/IIITH/LMA_Project_Phase1_data/LMA_Assignment1/nasa_systems_engineering_handbook_0.pdf",
  "chunk_count": 770,
  "question_count": 10,
  "baseline_successful": 10,
  "reranked_successful": 10,
  "chunking_seconds": 2.1084911589998683,
  "model_loading_seconds": 78.29831977999993,
  "embedding_preparation_seconds": 3.902144595999971,
  "elapsed_seconds_this_invocation": 183.15508630299996,
  "cuda_memory_at_end": {
    "allocated_gb": 2.101,
    "reserved_gb": 2.506,
    "peak_allocated_gb": 2.326,
    "peak_reserved_gb": 2.506
  },
  "mean_baseline_token_f1": 0.4164580034015774,
  "mean_reranked_toke

,question_id,failure_type,baseline_answer,reranked_answer,reference_answer,baseline_token_f1,reranked_token_f1,baseline_top3_recall,reranked_top3_recall,baseline_dense_retrieval_seconds,baseline_generation_seconds,rerank_seconds,reranked_generation_seconds,baseline_top3_chunk_ids,reranked_top3_chunk_ids
0,Q01,ordinary,The four System Design Processes described by ...,The four System Design Processes described by ...,The four System Design Processes are Stakehold...,0.477273,0.477273,None,None,0.030712,4.820219,0.031140,4.397533,p0053_c000|p0055_c000|p0015_c000,p0053_c000|p0055_c000|p0015_c000
1,Q02,keyword_trap,The Product Verification Process proves that t...,The Product Verification Process confirms that...,Verification demonstrates that an end product ...,0.461538,0.463415,None,None,0.008036,2.819006,0.033343,4.131143,p0198_c000|p0109_c000|p0109_c001,p0021_c000|p0099_c000|p0098_c001
2,Q03,ordinary,The product baseline is the approved technical...,The product baseline describes Level 1 Technic...,The product baseline describes the detailed ph...,0.243243,0.363636,None,None,0.025027,3.788617,0.042170,2.376389,p0157_c000|p0137_c000|p0098_c000,p0157_c000|p0156_c002|p0199_c000
3,Q04,rank_shift,The main steps in NASA's normal configuration ...,The main steps in NASA's normal configuration ...,The process includes preparing and submitting ...,0.427586,0.427586,None,None,0.008685,9.010513,0.033411,9.002809,p0158_c001|p0158_c000|p0155_c000,p0158_c001|p0158_c000|p0155_c000
4,Q05,exception,A major engineering change involves a modifica...,A major engineering change is an iteration in ...,A major engineering change significantly affec...,0.549020,0.627451,None,None,0.007840,5.222523,0.035628,5.326253,p0159_c000|p0191_c001|p0077_c002,p0159_c000|p0041_c000|p0019_c000
5,Q06,keyword_trap,The chair of NASA's internal Configuration Con...,The chair of NASA's internal Configuration Con...,The internal NASA Configuration Control Board ...,0.385542,0.385542,None,None,0.008217,3.672505,0.033431,3.593320,p0158_c000|p0158_c001|p0155_c000,p0158_c000|p0158_c001|p0155_c000
6,Q07,rank_shift,The allocated baseline is typically establishe...,The allocated baseline is typically establishe...,The allocated baseline is typically establishe...,0.578947,0.547945,None,None,0.007990,3.490665,0.032708,3.366214,p0156_c002|p0029_c001|p0156_c001,p0157_c000|p0156_c002|p0156_c001
7,Q08,ordinary,The as-deployed baseline is established after ...,The as-deployed baseline is not explicitly men...,The as-deployed baseline occurs at the Operati...,0.400000,0.390244,None,None,0.007452,2.390377,0.031755,4.242232,p0156_c001|p0029_c001|p0156_c002,p0156_c001|p0156_c002|p0157_c000
8,Q09,negation,NASA uses configuration change management to s...,NASA uses configuration change management to m...,Configuration change management controls appro...,0.405405,0.291667,None,None,0.007603,4.290232,0.033024,5.897528,p0158_c000|p0158_c001|p0155_c000,p0158_c000|p0154_c000|p0158_c001
9,Q10,rank_shift,When deciding the level of simulation fidelity...,When deciding the level of simulation fidelity...,"Factors include runtime constraints, the level...",0.236025,0.564706,None,None,0.011927,10.499374,0.033605,4.685858,p0109_c000|p0267_c000|p0021_c000,p0267_c000|p0021_c001|p0109_c000


,question_id,chunk_id,page,dense_rank,reranked_rank,dense_score,cross_encoder_score,relevant
0,Q01,p0053_c000,53,1,1,0.705196,7.089072,False
1,Q01,p0055_c000,55,2,2,0.703943,4.829300,False
2,Q01,p0015_c000,15,3,3,0.700558,4.173309,False
3,Q01,p0077_c000,77,8,4,0.674264,3.779458,False
4,Q01,p0086_c000,86,6,5,0.680717,3.453370,False
5,Q01,p0075_c000,75,10,6,0.667013,3.437336,False
6,Q01,p0008_c002,8,5,7,0.688856,1.613127,False
7,Q01,p0011_c000,11,7,8,0.678634,0.469509,False
8,Q01,p0251_c000,251,9,9,0.668929,-3.539368,False
9,Q01,p0253_c000,253,4,10,0.690429,-4.062927,False


## Interpreting the metrics and implementation

- **Bi-encoder / dense retrieval:** embeds the query and each chunk separately. The query vector is compared with stored chunk vectors; here, normalized vectors use dot product, equivalent to cosine similarity.
- **Cross-encoder:** processes a query and candidate chunk together and produces a relevance score. It can model token interactions more directly, but it is slower to apply to every chunk in a large corpus.
- **Initial top-k = 10:** the bi-encoder first narrows the search to ten candidates.
- **Final top-k = 3:** both pipelines send exactly three chunks to the answer model. Baseline uses the dense top three; re-ranking uses the top three after cross-encoder sorting.
- **Recall@k:** fraction of the question's known relevant chunks retrieved in the top k.
- **MRR@k:** reciprocal rank of the first relevant chunk, or zero if none appears in the top k.
- **nDCG@k:** rewards relevant chunks appearing earlier in the ranked list. This notebook uses binary relevance labels.
- **Token-overlap F1:** harmonic mean of token precision and recall against a reference answer. It is easy to compute, but it can penalize correct paraphrases and reward superficial overlap. Manually inspect answers; do not present it as a complete semantic correctness metric.
- **Latency:** timing is measured on the actual Colab runtime. This run does not automatically benchmark both GPUs; select A100 or T4 in Colab and run the same configuration on each, with the same questions and models, to compare.
- **GPU memory:** PyTorch peak allocation is reported where available. It is not a measure of total GPU use by every process.
- **No training is performed:** the embedding model, cross-encoder, and LLM are pretrained. This is a retrieval-and-generation experiment, so training/validation loss curves are not applicable unless you add a trained component.

## Expected output files

All files are saved under `OUTPUT_DIR` in Google Drive. The most important are `baseline_results.jsonl`, `reranked_results.jsonl`, `retrieval_logs.jsonl`, `reranking_logs.jsonl`, `answer_comparison_and_metrics.csv`, `rank_shift_analysis.csv`, `timings.jsonl`, `summary.json`, `run.log`, and the PNG plots.

**Fairness and limitations:** both approaches use the same chunk corpus, question, initial top-10 candidates, LLM, and prompt structure. Generation can still be stochastic if you change `TEMPERATURE`; keep it at zero for the comparison. The evaluation set is small and document-specific, so conclusions should be limited to this experiment.

In [ ]:

import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/RAG_Assignment/outputs")
CHUNK_FILE = OUT / "document_chunks.jsonl"

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

print("Total chunks:", len(chunks))
print("\nFirst chunk structure:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:2500])

keywords = {
    "Q01": ["Stakeholder Expectations Definition",
            "Technical Requirements Definition"],
    "Q02": ["Product Verification Process",
            "Product Validation Process"],
    "Q03": ["product baseline", "production acceptance test"],
    "Q04": ["NORMAL CONFIGURATION CHANGE PROCESS",
            "change request"],
    "Q05": ["major", "minor", "engineering change"],
    "Q06": ["Configuration Control Board", "change authority"],
    "Q07": ["Allocated Baseline", "Preliminary Design Review"],
    "Q08": ["As-Deployed Baseline", "Operational Readiness Review"],
    "Q09": ["Manage Configuration Change Control",
            "approved changes"],
    "Q10": ["Product Verification", "simulation fidelity"],
}

# Print matching chunks with their IDs and text.
for qid, terms in keywords.items():
    print(f"\n{'=' * 20} {qid} {'=' * 20}")
    matches = 0

    for chunk in chunks:
        text = str(chunk.get("text", chunk.get("content", "")))
        if any(term.lower() in text.lower() for term in terms):
            chunk_id = chunk.get(
                "chunk_id", chunk.get("id", "CHECK_CHUNK_SCHEMA")
            )
            print(f"\nChunk ID: {chunk_id}")
            print(text[:1200])
            matches += 1
            if matches >= 5:
                break

    if matches == 0:
        print("No matches found; inspect the chunk structure above.")


Total chunks: 770

First chunk structure:
{
  "chunk_id": "p0001_c000",
  "page": 1,
  "start_char": 0,
  "end_char": 118,
  "text": "National Aeronautics and Space Administration NASA SYSTEMS ENGINEERING HANDBOOK design test integrate fly www.nasa.gov"
}

==================== Q01 ====================

Chunk ID: p0003_c001
. . . . . . . 27 3.7 Project Phase D: System Assembly, Integration and Test, Launch. . . . . . 29 3.8 Project Phase E: Operations and Sustainment. . . . . . . . . . . . 31 3.9 Project Phase F: Closeout. . . . . . . . 31 3.10 Funding: The Budget Cycle. . . . . . . 33 3.11 Tailoring and Customization of NPR 7123. 1 Requirements . . . . . . . 34 3.11.1 Introduction 34 3.11.2 Criteria for Tailoring 34 3.11.3 Tailoring SE NPR Requirements Using the Compliance Matrix 35 3.11.4 Ways to Tailor a SE Requirement 36 3.11.5 Examples of Tailoring and Customization 37 3.11.6 Approvals for Tailoring 40 4.0 System Design Processes 43 4.1 Stakeholder Expectations Definition. . 4

In [ ]:

import json
from pathlib import Path

chunks_path = Path(
    "/content/drive/MyDrive/RAG_Assignment/outputs/document_chunks.jsonl"
)

with chunks_path.open("r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

for chunk_id in [
    "p0159_c000",
    "p0159_c001",
    "p0267_c000",
    "p0267_c001",
    "p0268_c000",
]:
    chunk = chunks_by_id.get(chunk_id)

    print("\n" + "=" * 80)
    if chunk is None:
        print(f"{chunk_id}: NOT FOUND")
        continue

    print(f'ID: {chunk["chunk_id"]} | Page: {chunk["page"]}')
    print(chunk["text"])



ID: p0159_c000 | Page: 159
149 6.0 Crosscutting Technical Management NASA SYSTEMS ENGINEERING HANDBOOK TYPES OF CONFIGURATION MANAGEMENT CHANGES • Engineering Change: An engineering change is an iteration in the baseline. Changes can be major or minor. They may or may not include a specification change. Changes affecting an external interface must be coordinated and approved by all stakeholders affected. > A “major” change is a change to the baseline configuration documentation that has significant impact (i.e., requires retrofit of delivered products or affects the baseline specification, cost, safety, compatibility with interfacing products, or operator, or maintenance training). > A ”minor” change corrects or modifies configuration documentation or processes without impact to the interchangeability of products or system elements in the system structure. • Waiver: A waiver is a documented agreement intentionally releasing a program or project from meeting a requirement. (Some Center

In [ ]:

from pathlib import Path
import json

OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

print("Output directory exists:", OUTPUT_DIR.exists())

for filename in [
    "evaluation_questions.json",
    "baseline_results.jsonl",
    "reranked_results.jsonl",
    "retrieval_logs.jsonl",
    "reranking_logs.jsonl",
    "rank_shift_analysis.csv",
    "answer_comparison_and_metrics.csv",
]:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

# Check whether the saved question file contains the updated relevance labels.
questions_path = OUTPUT_DIR / "evaluation_questions.json"

if questions_path.exists():
    saved_questions = json.loads(questions_path.read_text(encoding="utf-8"))
    print("\nSaved question relevance labels:")
    for q in saved_questions:
        print(q["id"], "->", q.get("relevant_chunk_ids", []))


Output directory exists: True
evaluation_questions.json: FOUND
baseline_results.jsonl: FOUND
reranked_results.jsonl: FOUND
retrieval_logs.jsonl: FOUND
reranking_logs.jsonl: FOUND
rank_shift_analysis.csv: FOUND
answer_comparison_and_metrics.csv: FOUND

Saved question relevance labels:
Q01 -> []
Q02 -> []
Q03 -> []
Q04 -> []
Q05 -> []
Q06 -> []
Q07 -> []
Q08 -> []
Q09 -> []
Q10 -> []


In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")
questions_path = OUTPUT_DIR / "evaluation_questions.json"

# Confirm the in-memory questions have the updated labels.
for q in QUESTIONS:
    print(q["id"], "->", q.get("relevant_chunk_ids", []))

# Stop if the current QUESTIONS list still has empty labels.
empty_ids = [
    q["id"] for q in QUESTIONS
    if not q.get("relevant_chunk_ids")
]
if empty_ids:
    raise ValueError(
        f"These questions still have empty relevant_chunk_ids: {empty_ids}. "
        "Update Cell 3A and run it first."
    )

# Save the updated questions.
questions_path.write_text(
    json.dumps(QUESTIONS, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nUpdated:", questions_path)

# Verify the saved file.
saved_questions = json.loads(
    questions_path.read_text(encoding="utf-8")
)
for q in saved_questions:
    print(q["id"], "->", q["relevant_chunk_ids"])

Q01 -> ['p0016_c000']
Q02 -> ['p0021_c00', 'p0021_c001']
Q03 -> ['p0157_c001']
Q04 -> ['p0158_c001']
Q05 -> ['p0159_c000']
Q06 -> ['p0158_c000']
Q07 -> ['p0157_c000']
Q08 -> ['p0158_c000']
Q09 -> ['p0158_c000']
Q10 -> ['p0267_c000', 'p0267_c001']

Updated: /content/drive/MyDrive/RAG_Assignment/outputs/evaluation_questions.json
Q01 -> ['p0016_c000']
Q02 -> ['p0021_c00', 'p0021_c001']
Q03 -> ['p0157_c001']
Q04 -> ['p0158_c001']
Q05 -> ['p0159_c000']
Q06 -> ['p0158_c000']
Q07 -> ['p0157_c000']
Q08 -> ['p0158_c000']
Q09 -> ['p0158_c000']
Q10 -> ['p0267_c000', 'p0267_c001']


In [ ]:

import json
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

# Fix Q02's chunk ID in the saved question file.
questions_path = OUTPUT_DIR / "evaluation_questions.json"
questions = json.loads(questions_path.read_text(encoding="utf-8"))

for q in questions:
    if q["id"] == "Q02":
        q["relevant_chunk_ids"] = ["p0021_c000", "p0021_c001"]

questions_path.write_text(
    json.dumps(questions, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Fixed Q02 relevance labels.")

# Inspect the saved retrieval log structure.
for filename in [
    "retrieval_logs.jsonl",
    "reranking_logs.jsonl",
    "baseline_results.jsonl",
    "reranked_results.jsonl",
]:
    path = OUTPUT_DIR / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if not path.exists():
        print("File not found.")
        continue

    with path.open("r", encoding="utf-8") as f:
        first_line = next((line for line in f if line.strip()), None)

    if first_line:
        record = json.loads(first_line)
        print("Fields:", list(record.keys()))
        print(json.dumps(record, ensure_ascii=False, indent=2)[:2500])
    else:
        print("File is empty.")


Fixed Q02 relevance labels.

==================== retrieval_logs.jsonl ====================
Fields: ['timestamp_utc', 'question_id', 'question', 'candidates']
{
  "timestamp_utc": "2026-10-19T09:09:54.339490+00:00",
  "question_id": "Q01",
  "question": "What are the four System Design Processes described by NASA, and what is their overall purpose?",
  "candidates": [
    {
      "chunk_id": "p0053_c000",
      "page": 53,
      "dense_rank": 1,
      "dense_score": 0.7051958441734314,
      "text": "43 NASA SYSTEMS ENGINEERING HANDBOOK 4.0 System Design Processes T his chapter describes the activities in the system design processes listed in FIGURE 2.1-1. The chap­ ter is separated into sections corresponding to pro­ cesses 1 to 4 listed in FIGURE 2.1-1. The tasks within each process are discussed in terms of inputs, activ­ ities, and outputs. Additional guidance is provided using examples that are relevant to NASA projects. The system design processes are interdependent, highly itera

In [ ]:

import json
import math
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

def read_jsonl_file(filename):
    path = OUTPUT_DIR / filename
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Load updated relevance labels and existing retrieval logs.
questions = json.loads(
    (OUTPUT_DIR / "evaluation_questions.json").read_text(encoding="utf-8")
)
baseline_logs = read_jsonl_file("retrieval_logs.jsonl")
rerank_logs = read_jsonl_file("reranking_logs.jsonl")

# If a log contains repeated entries for a question, keep its latest entry.
def latest_by_question(records):
    latest = {}
    for record in records:
        qid = record["question_id"]
        if (qid not in latest or
            record.get("timestamp_utc", "") >
            latest[qid].get("timestamp_utc", "")):
            latest[qid] = record
    return latest

baseline_by_q = latest_by_question(baseline_logs)
rerank_by_q = latest_by_question(rerank_logs)

def ordered_candidates(record, rank_field):
    return sorted(
        record["candidates"],
        key=lambda c: c.get(rank_field, 10**9)
    )

def metrics_at_k(ranked_ids, relevant_ids, k=3):
    relevant = set(relevant_ids)
    ranked = ranked_ids[:k]

    if not relevant:
        return {"recall": None, "mrr": None, "ndcg": None}

    hits = [1 if cid in relevant else 0 for cid in ranked]
    recall = len(set(ranked) & relevant) / len(relevant)

    reciprocal_rank = next(
        (1 / rank for rank, hit in enumerate(hits, 1) if hit), 0.0
    )

    dcg = sum(
        hit / math.log2(rank + 1)
        for rank, hit in enumerate(hits, 1)
    )
    ideal_count = min(len(relevant), k)
    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )
    ndcg = dcg / idcg if idcg else 0.0

    return {
        "recall": recall,
        "mrr": reciprocal_rank,
        "ndcg": ndcg
    }

metric_rows = []
rank_rows = []

for q in questions:
    qid = q["id"]
    relevant_ids = q.get("relevant_chunk_ids", [])

    if qid not in baseline_by_q or qid not in rerank_by_q:
        print(f"WARNING: Missing saved logs for {qid}")
        continue

    baseline_candidates = ordered_candidates(
        baseline_by_q[qid], "dense_rank"
    )
    reranked_candidates = ordered_candidates(
        rerank_by_q[qid], "reranked_rank"
    )

    baseline_ids = [c["chunk_id"] for c in baseline_candidates]
    reranked_ids = [c["chunk_id"] for c in reranked_candidates]

    bm = metrics_at_k(baseline_ids, relevant_ids, k=3)
    rm = metrics_at_k(reranked_ids, relevant_ids, k=3)

    metric_rows.append({
        "question_id": qid,
        "failure_type": q.get("failure_type", "ordinary"),
        "relevant_chunk_ids": ", ".join(relevant_ids),
        "baseline_top3": ", ".join(baseline_ids[:3]),
        "reranked_top3": ", ".join(reranked_ids[:3]),
        "baseline_recall_at3": bm["recall"],
        "reranked_recall_at3": rm["recall"],
        "baseline_mrr_at3": bm["mrr"],
        "reranked_mrr_at3": rm["mrr"],
        "baseline_ndcg_at3": bm["ndcg"],
        "reranked_ndcg_at3": rm["ndcg"],
    })

    # Track each verified relevant chunk across the original top-10 list.
    for cid in relevant_ids:
        b_rank = baseline_ids.index(cid) + 1 if cid in baseline_ids else None
        r_rank = reranked_ids.index(cid) + 1 if cid in reranked_ids else None

        rank_rows.append({
            "question_id": qid,
            "chunk_id": cid,
            "baseline_dense_rank": b_rank,
            "reranked_rank": r_rank,
            "baseline_in_top3": b_rank is not None and b_rank <= 3,
            "reranked_in_top3": r_rank is not None and r_rank <= 3,
            "rank_shift": (
                b_rank - r_rank
                if b_rank is not None and r_rank is not None
                else None
            ),
        })

metrics_df = pd.DataFrame(metric_rows)
rank_df = pd.DataFrame(rank_rows)

metrics_path = OUTPUT_DIR / "retrieval_metrics_recomputed.csv"
rank_path = OUTPUT_DIR / "rank_shift_analysis_recomputed.csv"

metrics_df.to_csv(metrics_path, index=False)
rank_df.to_csv(rank_path, index=False)

print("Per-question retrieval metrics:")
display(metrics_df)

print("\nRelevant-chunk rank shifts:")
display(rank_df)

metric_cols = [
    "baseline_recall_at3", "reranked_recall_at3",
    "baseline_mrr_at3", "reranked_mrr_at3",
    "baseline_ndcg_at3", "reranked_ndcg_at3"
]
print("\nMean retrieval metrics across questions:")
display(metrics_df[metric_cols].mean().to_frame("mean").T)

print("\nSaved:")
print(metrics_path)
print(rank_path)


Per-question retrieval metrics:


,question_id,failure_type,relevant_chunk_ids,baseline_top3,reranked_top3,baseline_recall_at3,reranked_recall_at3,baseline_mrr_at3,reranked_mrr_at3,baseline_ndcg_at3,reranked_ndcg_at3
0,Q01,ordinary,p0016_c000,"p0053_c000, p0055_c000, p0015_c000","p0053_c000, p0055_c000, p0015_c000",0.0,0.0,0.0,0.0,0.000000,0.000000
1,Q02,keyword_trap,"p0021_c000, p0021_c001","p0198_c000, p0109_c000, p0109_c001","p0021_c000, p0099_c000, p0098_c001",0.0,0.5,0.0,1.0,0.000000,0.613147
2,Q03,ordinary,p0157_c001,"p0157_c000, p0137_c000, p0098_c000","p0157_c000, p0156_c002, p0199_c000",0.0,0.0,0.0,0.0,0.000000,0.000000
3,Q04,rank_shift,p0158_c001,"p0158_c001, p0158_c000, p0155_c000","p0158_c001, p0158_c000, p0155_c000",1.0,1.0,1.0,1.0,1.000000,1.000000
4,Q05,exception,p0159_c000,"p0159_c000, p0191_c001, p0077_c002","p0159_c000, p0041_c000, p0019_c000",1.0,1.0,1.0,1.0,1.000000,1.000000
5,Q06,keyword_trap,p0158_c000,"p0158_c000, p0158_c001, p0155_c000","p0158_c000, p0158_c001, p0155_c000",1.0,1.0,1.0,1.0,1.000000,1.000000
6,Q07,rank_shift,p0157_c000,"p0156_c002, p0029_c001, p0156_c001","p0157_c000, p0156_c002, p0156_c001",0.0,1.0,0.0,1.0,0.000000,1.000000
7,Q08,ordinary,p0158_c000,"p0156_c001, p0029_c001, p0156_c002","p0156_c001, p0156_c002, p0157_c000",0.0,0.0,0.0,0.0,0.000000,0.000000
8,Q09,negation,p0158_c000,"p0158_c000, p0158_c001, p0155_c000","p0158_c000, p0154_c000, p0158_c001",1.0,1.0,1.0,1.0,1.000000,1.000000
9,Q10,rank_shift,"p0267_c000, p0267_c001","p0109_c000, p0267_c000, p0021_c000","p0267_c000, p0021_c001, p0109_c000",0.5,0.5,0.5,1.0,0.386853,0.613147



Relevant-chunk rank shifts:


,question_id,chunk_id,baseline_dense_rank,reranked_rank,baseline_in_top3,reranked_in_top3,rank_shift
0,Q01,p0016_c000,NaN,NaN,False,False,NaN
1,Q02,p0021_c000,4.0,1.0,False,True,3.0
2,Q02,p0021_c001,NaN,NaN,False,False,NaN
3,Q03,p0157_c001,NaN,NaN,False,False,NaN
4,Q04,p0158_c001,1.0,1.0,True,True,0.0
5,Q05,p0159_c000,1.0,1.0,True,True,0.0
6,Q06,p0158_c000,1.0,1.0,True,True,0.0
7,Q07,p0157_c000,6.0,1.0,False,True,5.0
8,Q08,p0158_c000,NaN,NaN,False,False,NaN
9,Q09,p0158_c000,1.0,1.0,True,True,0.0



Mean retrieval metrics across questions:


,baseline_recall_at3,reranked_recall_at3,baseline_mrr_at3,reranked_mrr_at3,baseline_ndcg_at3,reranked_ndcg_at3
mean,0.45,0.6,0.45,0.7,0.438685,0.622629



Saved:
/content/drive/MyDrive/RAG_Assignment/outputs/retrieval_metrics_recomputed.csv
/content/drive/MyDrive/RAG_Assignment/outputs/rank_shift_analysis_recomputed.csv


In [ ]:

import json
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

def load_jsonl(filename):
    with (OUTPUT_DIR / filename).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

baseline_logs = load_jsonl("retrieval_logs.jsonl")
rerank_logs = load_jsonl("reranking_logs.jsonl")

questions_file = OUTPUT_DIR / "evaluation_questions.json"
questions = json.loads(questions_file.read_text(encoding="utf-8"))

# Keep the latest saved log entry for each question.
def latest_records(records):
    result = {}
    for record in records:
        qid = record["question_id"]
        if (
            qid not in result
            or record.get("timestamp_utc", "")
            > result[qid].get("timestamp_utc", "")
        ):
            result[qid] = record
    return result

baseline_by_q = latest_records(baseline_logs)
rerank_by_q = latest_records(rerank_logs)
questions_by_id = {q["id"]: q for q in questions}

for qid in ["Q01", "Q03", "Q08"]:
    q = questions_by_id[qid]

    print("\n" + "=" * 90)
    print(qid, "-", q["question"])
    print("Currently labelled relevant chunks:", q["relevant_chunk_ids"])

    for label, records, rank_field in [
        ("BASELINE DENSE RETRIEVAL", baseline_by_q, "dense_rank"),
        ("AFTER CROSS-ENCODER RERANKING", rerank_by_q, "reranked_rank"),
    ]:
        print(f"\n--- {label}: top 10 ---")
        candidates = sorted(
            records[qid]["candidates"],
            key=lambda c: c.get(rank_field, 999)
        )

        for c in candidates[:10]:
            print(
                f"\nRank {c.get(rank_field)} | "
                f"{c['chunk_id']} | Page {c['page']}"
            )
            print(c["text"][:650].replace("\n", " "))

    # Show the actual text of the currently labelled chunks.
    print("\n--- CURRENTLY LABELLED CHUNKS ---")
    candidate_lookup = {
        c["chunk_id"]: c
        for c in (
            baseline_by_q[qid]["candidates"]
            + rerank_by_q[qid]["candidates"]
        )
    }

    for cid in q["relevant_chunk_ids"]:
        if cid in candidate_lookup:
            print(f"\n{cid}: {candidate_lookup[cid]['text'][:650]}")
        else:
            print(
                f"\n{cid}: Not in this question's retrieved top 10. "
                "Its text can be checked in document_chunks.jsonl."
            )



Q01 - What are the four System Design Processes described by NASA, and what is their overall purpose?
Currently labelled relevant chunks: ['p0016_c000']

--- BASELINE DENSE RETRIEVAL: top 10 ---

Rank 1 | p0053_c000 | Page 53
43 NASA SYSTEMS ENGINEERING HANDBOOK 4.0 System Design Processes T his chapter describes the activities in the system design processes listed in FIGURE 2.1-1. The chap­ ter is separated into sections corresponding to pro­ cesses 1 to 4 listed in FIGURE 2.1-1. The tasks within each process are discussed in terms of inputs, activ­ ities, and outputs. Additional guidance is provided using examples that are relevant to NASA projects. The system design processes are interdependent, highly iterative and recursive processes resulting in a validated set of requirements and a design solution that satisfies a set of stakeholder expectations. There are 

Rank 2 | p0055_c000 | Page 55
45 4.0 System Design Processes NASA SYSTEMS ENGINEERING HANDBOOK the next level of decompos

In [ ]:

import json
from pathlib import Path

chunks_path = Path(
    "/content/drive/MyDrive/RAG_Assignment/outputs/document_chunks.jsonl"
)

with chunks_path.open("r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

chunks_by_id = {c["chunk_id"]: c for c in chunks}

for cid in [
    "p0016_c000",
    "p0053_c000",
    "p0157_c000",
    "p0157_c001",
    "p0158_c000",
]:
    print("\n" + "=" * 80)
    chunk = chunks_by_id.get(cid)
    if chunk:
        print(f"Chunk: {cid} | PDF page: {chunk['page']}")
        print(chunk["text"])
    else:
        print(f"{cid} was not found.")



Chunk: p0016_c000 | PDF page: 16
6 2.0 Fundamentals of Systems Engineering NASA SYSTEMS ENGINEERING HANDBOOK Requirements Definition Processes 1. Stakeholders Expectations Definition 2. Technical Requirements Definition 3. Logical Decomposition 4. Design Solution Definition System Design Processes Requirements Flow Down from Level above Technical Management Processes 10. Technical Planning 11. Requirement Management 12. Interface Management 13. Technical Risk Management 14. Configuration Management 15. Technical Data Management 16. Technical Assessment Technical Decision Analysis Process 17. Decision Analysis 9. Product Transition 8. Product Validation 7. Product Verification 6. Product Integration 5. Product Implementation Product Realization Processes System Design Processes applied to each product layer down through system structure Product Realization Processes applied to each product layer up through system structure Requirements Flow Down To Level below Realized Products to Leve

In [ ]:

import json
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/RAG_Assignment/outputs"
)

questions_path = OUTPUT_DIR / "evaluation_questions.json"

with open(questions_path, "r", encoding="utf-8") as f:
    questions = json.load(f)

# Update only the three relevance labels we reviewed.
updated_labels = {
    "Q01": ["p0053_c000"],
    "Q03": ["p0157_c000", "p0157_c001"],
    "Q08": ["p0158_c000"],
}

for q in questions:
    if q["id"] in updated_labels:
        q["relevant_chunk_ids"] = updated_labels[q["id"]]

with open(questions_path, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)

print("Updated relevance labels:")
for q in questions:
    if q["id"] in updated_labels:
        print(q["id"], "->", q["relevant_chunk_ids"])

print("\nSaved to:", questions_path)


Updated relevance labels:
Q01 -> ['p0053_c000']
Q03 -> ['p0157_c000', 'p0157_c001']
Q08 -> ['p0158_c000']

Saved to: /content/drive/MyDrive/RAG_Assignment/outputs/evaluation_questions.json


In [ ]:

import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

with open(OUT / "evaluation_questions.json", encoding="utf-8") as f:
    questions = json.load(f)

chunks = {}
with open(OUT / "document_chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        chunks[item["chunk_id"]] = item

for q in questions:
    if q["id"] in {"Q01", "Q03", "Q08"}:
        continue

    print("\n" + "=" * 90)
    print(q["id"], "-", q.get("question", q.get("query", "")))
    print("Current relevant IDs:", q.get("relevant_chunk_ids", []))

    for chunk_id in q.get("relevant_chunk_ids", []):
        chunk = chunks.get(chunk_id)
        print(f"\n--- {chunk_id} ---")

        if chunk is None:
            print("WARNING: Chunk ID not found!")
        else:
            print(chunk.get("text", chunk.get("chunk_text", "")))



Q02 - What is the difference between the Product Verification Process and the Product Validation Process?
Current relevant IDs: ['p0021_c000', 'p0021_c001']

--- p0021_c000 ---
11 2.0 Fundamentals of Systems Engineering NASA SYSTEMS ENGINEERING HANDBOOK 2.4 Distinctions between Product Verification and Product Validation From a process perspective, the Product Verification and Product Validation processes may be similar in nature, but the objectives are fundamentally different: • Verification of a product shows proof of compli­ ance with requirements—that the product can meet each “shall” statement as proven though per­ formance of a test, analysis, inspection, or demon­ stration (or combination of these). • Validation of a product shows that the prod­ uct accomplishes the intended purpose in the intended environment—that it meets the expec­ tations of the customer and other stakeholders as shown through performance of a test, analysis, inspection, or demonstration. Verification testi

In [ ]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/RAG_Assignment/outputs/document_chunks.jsonl"
)

target_ids = {"p0156_c001", "p0156_c002", "p0157_c000"}

with open(path, encoding="utf-8") as f:
    for line in f:
        chunk = json.loads(line)
        if chunk.get("chunk_id") in target_ids:
            print("\n" + "=" * 80)
            print("Chunk:", chunk["chunk_id"])
            print(chunk.get("text", chunk.get("chunk_text", "")))



Chunk: p0156_c001
iers for a product and its configuration documentation. The CM activity associated with identification includes selecting the Configuration Items (CIs), determin­ ing the CIs’ associated configuration documentation, determining the appropriate change control author­ ity, issuing unique identifiers for both CIs and CI documentation, releasing configuration documenta­ tion, and establishing configuration baselines. NASA has four baselines, each of which defines a dis­ tinct phase in the evolution of a product design. The baseline identifies an agreed-to description of attri­ butes of a CI at a point in time and provides a known configuration to which changes are addressed. Baselines are established by agreeing to (and docu­ menting) the stated definition of a CI’s attributes. The approved “current” baseline defines the basis of the subsequent change. The system specification is typically finalized following the SRR. The functional baseline is established at the SDR and

In [ ]:

import json
from pathlib import Path

out = Path("/content/drive/MyDrive/RAG_Assignment/outputs")
questions_path = out / "evaluation_questions.json"

with open(questions_path, encoding="utf-8") as f:
    questions = json.load(f)

final_labels = {
    "Q01": ["p0053_c000"],
    "Q02": ["p0021_c000", "p0021_c001"],
    "Q03": ["p0157_c000", "p0157_c001"],
    "Q04": ["p0158_c001"],
    "Q05": ["p0159_c000"],
    "Q06": ["p0158_c000"],
    "Q07": ["p0156_c002", "p0157_c000"],
    "Q08": ["p0158_c000"],
    "Q09": ["p0158_c000"],
    "Q10": ["p0267_c000", "p0267_c001"],
}

for q in questions:
    qid = q["id"]
    if qid in final_labels:
        q["relevant_chunk_ids"] = final_labels[qid]

with open(questions_path, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)

for q in questions:
    print(q["id"], "->", q.get("relevant_chunk_ids", []))

print("\nSaved validated labels to:", questions_path)


Q01 -> ['p0053_c000']
Q02 -> ['p0021_c000', 'p0021_c001']
Q03 -> ['p0157_c000', 'p0157_c001']
Q04 -> ['p0158_c001']
Q05 -> ['p0159_c000']
Q06 -> ['p0158_c000']
Q07 -> ['p0156_c002', 'p0157_c000']
Q08 -> ['p0158_c000']
Q09 -> ['p0158_c000']
Q10 -> ['p0267_c000', 'p0267_c001']

Saved validated labels to: /content/drive/MyDrive/RAG_Assignment/outputs/evaluation_questions.json


In [ ]:

import json
from pathlib import Path
import numpy as np
import pandas as pd

OUT = Path("/content/drive/MyDrive/RAG_Assignment/outputs")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

with open(OUT / "evaluation_questions.json", encoding="utf-8") as f:
    questions = json.load(f)

retrieval_logs = read_jsonl(OUT / "retrieval_logs.jsonl")
reranking_logs = read_jsonl(OUT / "reranking_logs.jsonl")

def get_qid(row):
    for key in ("question_id", "qid", "id"):
        if key in row and str(row[key]).startswith("Q"):
            return str(row[key])
    return None

def get_chunk_id(item):
    if isinstance(item, str):
        return item
    for key in ("chunk_id", "id"):
        if key in item:
            return item[key]
    return None

def get_ranked_chunks(row, rank_field):
    # Logs may store ranked chunks as a list of objects.
    for key in ("results", "ranked_results", "retrieved_chunks",
                "candidates", "chunks", "top_k"):
        value = row.get(key)
        if isinstance(value, list):
            items = sorted(
                value,
                key=lambda x: x.get(rank_field, x.get("rank", 999))
                if isinstance(x, dict) else 999
            )
            ids = [get_chunk_id(x) for x in items]
            ids = [x for x in ids if x is not None]
            if ids:
                return ids

    # Or the log may already contain ranked chunk IDs.
    for key in ("ranked_chunk_ids", "chunk_ids", "top_chunk_ids"):
        value = row.get(key)
        if isinstance(value, list):
            return [get_chunk_id(x) for x in value]

    return []

def latest_by_question(rows):
    grouped = {}
    for row in rows:
        qid = get_qid(row)
        if qid:
            grouped[qid] = row  # retain latest occurrence
    return grouped

base_by_q = latest_by_question(retrieval_logs)
rerank_by_q = latest_by_question(reranking_logs)

def metrics(ranked, relevant, k=3):
    ranked = ranked[:k]
    relevant = set(relevant)
    if not relevant:
        return np.nan, np.nan, np.nan

    hits = [1 if chunk in relevant else 0 for chunk in ranked]
    recall = sum(hits) / len(relevant)

    rr = next((1 / (i + 1) for i, hit in enumerate(hits) if hit), 0.0)

    dcg = sum(hit / np.log2(i + 2) for i, hit in enumerate(hits))
    ideal_hits = [1] * min(len(relevant), k)
    idcg = sum(hit / np.log2(i + 2)
               for i, hit in enumerate(ideal_hits))
    ndcg = dcg / idcg if idcg else 0.0

    return recall, rr, ndcg

rows = []

for q in questions:
    qid = q["id"]
    relevant = q.get("relevant_chunk_ids", [])

    b_row = base_by_q.get(qid, {})
    r_row = rerank_by_q.get(qid, {})

    # Adapt to the rank fields used in the saved logs.
    baseline = get_ranked_chunks(b_row, "dense_rank")
    reranked = get_ranked_chunks(r_row, "reranked_rank")

    b = metrics(baseline, relevant)
    r = metrics(reranked, relevant)

    rows.append({
        "question_id": qid,
        "baseline_recall@3": b[0],
        "reranked_recall@3": r[0],
        "baseline_mrr@3": b[1],
        "reranked_mrr@3": r[1],
        "baseline_ndcg@3": b[2],
        "reranked_ndcg@3": r[2],
        "baseline_top3": baseline[:3],
        "reranked_top3": reranked[:3],
    })

df = pd.DataFrame(rows)

metric_cols = [
    "baseline_recall@3", "reranked_recall@3",
    "baseline_mrr@3", "reranked_mrr@3",
    "baseline_ndcg@3", "reranked_ndcg@3"
]

print("MEAN RETRIEVAL METRICS")
display(df[metric_cols].mean().to_frame("Mean").round(4))

print("\nPER-QUESTION SCORES")
display(df[["question_id"] + metric_cols].round(4))

print("\nTOP-3 RANKINGS")
display(df[["question_id", "baseline_top3", "reranked_top3"]])

df.to_csv(OUT / "retrieval_metrics_recomputed.csv", index=False)
print("\nSaved:", OUT / "retrieval_metrics_recomputed.csv")


MEAN RETRIEVAL METRICS


,Mean
baseline_recall@3,0.6500
reranked_recall@3,0.7500
baseline_mrr@3,0.7500
reranked_mrr@3,0.9000
baseline_ndcg@3,0.6613
reranked_ndcg@3,0.7839



PER-QUESTION SCORES


,question_id,baseline_recall@3,reranked_recall@3,baseline_mrr@3,reranked_mrr@3,baseline_ndcg@3,reranked_ndcg@3
0,Q01,1.0,1.0,1.0,1.0,1.0000,1.0000
1,Q02,0.0,0.5,0.0,1.0,0.0000,0.6131
2,Q03,0.5,0.5,1.0,1.0,0.6131,0.6131
3,Q04,1.0,1.0,1.0,1.0,1.0000,1.0000
4,Q05,1.0,1.0,1.0,1.0,1.0000,1.0000
5,Q06,1.0,1.0,1.0,1.0,1.0000,1.0000
6,Q07,0.5,1.0,1.0,1.0,0.6131,1.0000
7,Q08,0.0,0.0,0.0,0.0,0.0000,0.0000
8,Q09,1.0,1.0,1.0,1.0,1.0000,1.0000
9,Q10,0.5,0.5,0.5,1.0,0.3869,0.6131



TOP-3 RANKINGS


,question_id,baseline_top3,reranked_top3
0,Q01,"[p0053_c000, p0055_c000, p0015_c000]","[p0053_c000, p0055_c000, p0015_c000]"
1,Q02,"[p0198_c000, p0109_c000, p0109_c001]","[p0021_c000, p0099_c000, p0098_c001]"
2,Q03,"[p0157_c000, p0137_c000, p0098_c000]","[p0157_c000, p0156_c002, p0199_c000]"
3,Q04,"[p0158_c001, p0158_c000, p0155_c000]","[p0158_c001, p0158_c000, p0155_c000]"
4,Q05,"[p0159_c000, p0191_c001, p0077_c002]","[p0159_c000, p0041_c000, p0019_c000]"
5,Q06,"[p0158_c000, p0158_c001, p0155_c000]","[p0158_c000, p0158_c001, p0155_c000]"
6,Q07,"[p0156_c002, p0029_c001, p0156_c001]","[p0157_c000, p0156_c002, p0156_c001]"
7,Q08,"[p0156_c001, p0029_c001, p0156_c002]","[p0156_c001, p0156_c002, p0157_c000]"
8,Q09,"[p0158_c000, p0158_c001, p0155_c000]","[p0158_c000, p0154_c000, p0158_c001]"
9,Q10,"[p0109_c000, p0267_c000, p0021_c000]","[p0267_c000, p0021_c001, p0109_c000]"



Saved: /content/drive/MyDrive/RAG_Assignment/outputs/retrieval_metrics_recomputed.csv
